# SD1.5 Prompt-Mismatched In-Range Recovery Results

This notebook recreates the paper-style recovery figures for the in-range prompt-mismatched experiment. The target image is still in the beach-style distribution, but the sampling prior and recovery prompt may disagree, so the figures show how reconstruction quality changes when the prior used to acquire measurements is not the same as the prompt used during recovery.

Run it top to bottom after the prompt-mismatched suites are available under `results/`. The next cell tries the full split run tags first (`first4` plus `last3`), then unsplit/direct prompt-mismatched tags. If no prompt-mismatched rows are found, the notebook leaves the analysis table empty and prints a reminder to run the suites first.

In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
for search_root in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_dir = search_root / 'analyze_results'
    helper_path = helper_dir / 'sd15_recovery_analysis.py'
    if helper_path.exists():
        if str(helper_dir) not in sys.path:
            sys.path.insert(0, str(helper_dir))
        break
    for child in search_root.iterdir():
        if not child.is_dir():
            continue
        helper_dir = child / 'analyze_results'
        helper_path = helper_dir / 'sd15_recovery_analysis.py'
        if helper_path.exists():
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_recovery_analysis.py from the notebook cwd.')

import sd15_recovery_analysis as recovery
recovery = importlib.reload(recovery)

SD15_ROOT = recovery.find_sd15_root(NOTEBOOK_DIR)
TAG_GROUP_CANDIDATES = recovery.split_tag_group_candidates('prompt_mismatched/sunset')
SAMPLING_METHODS = list(recovery.DEFAULT_SAMPLING_METHODS)
ALLOWED_SAMPLING_PERC = {0.00015625, 0.0003125, 0.000625, 0.00125, 0.0025, 0.005, 0.01}
EXCLUDED_SAMPLING_CONDITIONS = set()
OUTPUT_ROOT = SD15_ROOT / 'results' / 'figures'

analysis = recovery.load_recovery_analysis(
    SD15_ROOT,
    tag_group_candidates=TAG_GROUP_CANDIDATES,
    sampling_methods=SAMPLING_METHODS,
    allowed_sampling_percentages=ALLOWED_SAMPLING_PERC,
    excluded_sampling_conditions=EXCLUDED_SAMPLING_CONDITIONS,
    output_root=OUTPUT_ROOT,
)
ROWS = analysis.rows
MEAN_TABLE = analysis.mean_table
ACTIVE_TAG = analysis.active_tag
LOADED_TAGS = analysis.loaded_tags
OUTPUT_DIR = analysis.output_dir

print(f'Active tag: {ACTIVE_TAG}')
print(f'Loaded source tags: {LOADED_TAGS}')
print(f'Loaded {len(ROWS)} run rows.')
display(MEAN_TABLE)
if MEAN_TABLE.empty:
    print('No recovery rows found yet. Run the suite first, then rerun this notebook.')


## Metric Curves

This cell calls the shared recovery plotting helpers to export metric curves for `psnr_db`, `ssim`, and `pixel_mae` plus a combined PSNR/SSIM panel for each diffusion/sampling method that has rows. Each subplot fixes a sampling prior, the colored lines compare recovery prompts, and the x-axis is the sampling ratio `m/n` on a log scale.

Curves show the mean over repeats. The shaded region is a 95% normal-approximation confidence interval, computed as mean +/- 1.96 SEM in the plotted metric units. The black dashed reference is the zero-filled inverse FFT baseline when those metrics are present.

In [ ]:
METRIC_OUTPUTS = recovery.export_metric_figures(
    ROWS,
    OUTPUT_DIR,
    show=True,
)
METRIC_OUTPUTS


## Recovery Grid

This cell builds the image grids used to inspect reconstruction quality directly. For each sampling prior, it selects one target item and one sampling ratio, then shows the ground truth, the zero-filled inverse FFT baseline, and the best available reconstruction for each recovery prompt.

The "best" reconstruction in each tile is selected from the loaded rows by PSNR first and SSIM second, so the grid is a compact visual counterpart to the metric curves. The cell saves one PDF per sampling prior in `OUTPUT_DIR` and displays the figures inline.

In [ ]:
IMAGE_SAMPLING_PERC = 0.00125

GRID_OUTPUTS = recovery.export_recovery_grids(
    ROWS,
    SD15_ROOT,
    OUTPUT_DIR,
    sampling_method=SAMPLING_METHODS[0],
    sampling_percentage=IMAGE_SAMPLING_PERC,
    show=True,
)
GRID_OUTPUTS
